# AI应用基础（48课时版） · 工单M2-3a《MLP神经网络》
**第5周 | 2课时(80分钟) | 理实一体**  | 教师参考版（完成态·同构学生版）

| 层 | 任务定位 | 达标要求 |
|----|---------|---------|
| **A 师傅** | 进阶拓展 | 创新优化；示范讲解 |
| **B 组长** | 标准达成 | F1≥0.85 |
| **C 操作员** | 基础完成 | F1≥0.70且口径正确 |
| **D 学徒** | 跟做保底 | 跟做示例跑通TODO-1, 程序出结果 |
> 每4周按工单成绩动态升降层（连续2次≥B升一层）



## 一、课前资讯（Predict·预测）
> **预习快问**：['W6课前测①：本课工单M2-1a的及格线是什么？（答：F1≥0.70且口径正确）', "W6课前测②：'分层train/test'用一句话说说你的理解"]

> **课前检查**：导学单+预习自测已完成；data/数据已就位、starter可运行（--check环境OK）

## 二、PRIMM五步法（先读后写）
本工单按国际编程教学法 **PRIMM** 组织：
1. **Predict 预测**——先运行下方示例，猜输出再验证；
2. **Run 运行**——跑通环境与数据；
3. **Investigate 研究**——读懂starter每段在做什么；
4. **Modify 修改**——完成TODO改造；
5. **Make 创造**——独立完成工单任务。


In [ ]:
# --- 环境与数据准备(勿改, 运行即可) ---
import os, sys, subprocess
os.chdir(os.path.abspath("../"))          # 切到课程包根目录(data/所在处)
print("工作目录:", os.getcwd())


In [ ]:
# --- 教师版运行环境（完成态依赖，勿改）---
import json, re, time          # solutions 完成态所需
import numpy as np
PKG = os.getcwd()              # 课程包根（学生版环境cell已chdir至此）
print("教师版运行环境就绪 · PKG =", PKG)

## 三、任务要求（Make）
**实施步骤**
1. 回顾预习卡与微课
2. 知识精讲（先例后理）
3. 演示代码骨架
4. 基础实现（D/C层保底）
5. 小结+布置第2次课

**验收标准**：及格(D)=`F1≥0.70且口径正确` ；优秀(B+)=`F1≥0.85`


### Starter 1：`# -*- coding: utf-8 -*-`
（Investigate: 读代码→Modify: 补全TODO）

In [ ]:
# -*- coding: utf-8 -*-
"""W5 MLP神经网络 —— 学生starter（M2-3a）
数据: 合成振动频谱二分类(正常/磨损)；目标: 准确率≥75%+损失曲线诊断报告
"""
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

### Starter 2：`def make_spectra(n=600, seed=42):`
（Investigate: 读代码→Modify: 补全TODO）

In [ ]:
def make_spectra(n=600, seed=42):
    rng = np.random.RandomState(seed)
    X = rng.normal(0, 1, (n, 30))
    y = rng.binomial(1, .5, n)
    X[y == 1, 2:8] += rng.uniform(.8, 1.6)
    return X, y

### Starter 3：`def main():`
（Investigate: 读代码→Modify: 补全TODO）

In [ ]:
def main():
    X, y = make_spectra()
    Xtr, Xte, ytr, yte = train_test_split(X, y, stratify=y, random_state=42)
    sc = StandardScaler().fit(Xtr)
    Xtr, Xte = sc.transform(Xtr), sc.transform(Xte)

    # 【参考答案】TODO-1 训练MLP(64,32) + early_stopping(验证集自动从训练集切出)
    mlp = MLPClassifier(hidden_layer_sizes=(64, 32), early_stopping=True,
                        random_state=0, max_iter=300).fit(Xtr, ytr)
    # 【参考答案】TODO-2 测试准确率(目标≥0.75)
    acc = mlp.score(Xte, yte)
    print(f"测试准确率 = {acc:.3f}（目标≥0.75 → {'达标' if acc >= 0.75 else '未达标'}）"
          f"  实际训练轮数={len(mlp.loss_curve_)}（早停生效）")

    # 【参考答案】TODO-3 损失曲线诊断图, 标注过拟合区间, 存 diagnosis.png
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    curve = np.asarray(mlp.loss_curve_)
    half = len(curve) // 2
    fig, ax = plt.subplots(figsize=(6, 3.6))
    ax.plot(curve, lw=1.6, label="training loss")
    ax.axvspan(half, len(curve) - 1, color="tab:red", alpha=.12,
               label="后段斜率趋平(疑似过拟合/收敛区)")
    ax.set_xlabel("epoch"); ax.set_ylabel("loss")
    ax.set_title("M2-3a MLP 损失曲线诊断（early_stopping）")
    ax.legend(); fig.tight_layout(); fig.savefig("diagnosis.png", dpi=120)
    plt.close(fig)
    print(f"损失曲线: 首轮={curve[0]:.4f} 中点={curve[half]:.4f} 末轮={curve[-1]:.4f}"
          f" → {'仍在下降, 收敛良好' if curve[-1] < curve[half] else '进入平台期'}; 图存 diagnosis.png")

    # 【参考答案】TODO-4(优秀) sklearn无Dropout → 用 L2 正则 alpha=1e-3 作对照实验
    reg = MLPClassifier(hidden_layer_sizes=(64, 32), alpha=1e-3, early_stopping=True,
                        random_state=0, max_iter=300).fit(Xtr, ytr)
    r_acc = reg.score(Xte, yte)
    print(f"正则对照(alpha=1e-3): 测试准确率={r_acc:.3f}  较默认{'+' if r_acc >= acc else ''}"
          f"{(r_acc - acc) * 100:.1f}%")
    print("结论: 中频段(2-7维)抬升是可学信号; 结构+早停+正则三件套同时守住准确率与泛化。")

### 💡 分层提示（先自己试5分钟再看）
**思路提示**
> - TODO-1:
>   训练MLP(hidden_layer_sizes=(64,32),
>   early_stopping=True) TODO-2:
>   打印测试准确率(目标≥0.75) TODO-3: 取
>   mlp.loss_curve_ 画损失曲线, 标注过拟合区间, 存
>   diagnosis.png TODO-4(优秀):
>   Dropout替代实验——sklearn无Dropout,
>   用alpha=1e-3正则对比

| 层 | 这样做 |
|----|--------|
| **D 学徒** | 跟教师演示逐步照做：定位TODO行 → 按上面思路逐句翻译成代码 → 运行看输出 → 对不上就看下方"报错自查" |
| **C 操作员** | 只看思路提示独立实现；卡住再对照教师课堂演示 |
| **B/A 组长·师傅** | 不看提示直接做；完成后做扩展：改参数/换实现对比效果，写进工单"参数-收益"留痕 |

**报错自查三板斧**：①读最后一行错误信息 → ②检查变量名/缩进/括号 → ③还不行问同层同学或教师
**自检**：对照上方"验收标准"，及格(D)=`F1≥0.70且口径正确`，优秀(B+)=`F1≥0.85`

### Starter 4：`if __name__ == "__main__":`
（Investigate: 读代码→Modify: 补全TODO）

In [ ]:
from sklearn.model_selection import train_test_split

from sklearn.neural_network import MLPClassifier

from sklearn.preprocessing import StandardScaler

X, y = make_spectra()

Xtr, Xte, ytr, yte = train_test_split(X, y, stratify=y, random_state=42)

sc = StandardScaler().fit(Xtr)

Xtr, Xte = sc.transform(Xtr), sc.transform(Xte)

print("=== 工单 M2-3a MLP 神经网络 · 完成态 ===")

print(f"数据: {X.shape[0]} 样本 × {X.shape[1]} 维频谱 · 隐藏层 (64, 32) · 标准化已启用")

mlp = MLPClassifier(hidden_layer_sizes=(64, 32), early_stopping=True,
                    random_state=0, max_iter=300).fit(Xtr, ytr)

acc = mlp.score(Xte, yte)

curve = mlp.loss_curve_

print(f"训练轮数 {len(curve)}（early_stopping 提前停止）  测试准确率 = {acc:.3f}（目标≥0.75）")

print("损失曲线诊断:")

n = len(curve)

for tag, i in [("首轮", 0), ("1/4", n // 4), ("1/2", n // 2), ("末轮", n - 1)]:
    print(f"  {tag:<4} epoch{i + 1:>3}  loss={curve[i]:.4f}")

print(f"  收敛判断: 末轮loss {curve[-1]:.4f} {'持续下降→收敛良好' if curve[-1] < curve[n // 2] else '平台期→已收敛'}")

reg = MLPClassifier(hidden_layer_sizes=(64, 32), alpha=1e-3, early_stopping=True,
                    random_state=0, max_iter=300).fit(Xtr, ytr)

print(f"\n正则对照（alpha=1e-3 等价 L2 抑制过拟合）: 测试准确率 = {reg.score(Xte, yte):.3f}")

print("结论: 中频段抬升（2-7维）是可学信号；网络结构+早停+正则三件套，准确率与泛化同时守住。")

## 四、提交要求与避坑
- 提交: `学号_姓名_M2-3a.py` 或本notebook(含输出)
